In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

# Configuración de visualización para auditoría de datos
pd.set_option('display.max_columns', None)

# ==========================================
# 1. INGESTA DE DATOS (Data Ingestion)
# ==========================================
# Cargamos los archivos base. Nota: Mantenemos los nombres originales para trazabilidad.
train = pd.read_csv('/train.csv')
stores = pd.read_csv('/stores.csv')
oil = pd.read_csv('/oil.csv')
holidays = pd.read_csv('/holidays_events.csv')

# Convertir a datetime: Paso crítico para que el motor de Pandas reconozca la serie temporal
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

# ==========================================
# 2. INTEGRACIÓN (Merging Strategy)
# ==========================================
# Usamos 'left join' para no perder registros del set de ventas (nuestra fuente de verdad)
df = train.merge(stores, on='store_nbr', how='left')
df = df.merge(oil, on='date', how='left')

# Limpieza de feriados antes del merge para evitar duplicidad de filas (Fan-out effect)
holidays = holidays.drop_duplicates(subset=['date'], keep='first')
df = df.merge(holidays, on='date', how='left')

# ==========================================
# 3. LIMPIEZA Y CURACIÓN (Data Cleaning)
# ==========================================
# Renombramiento ejecutivo para evitar colisiones de nombres (type_x, type_y)
df.rename(columns={'type_x': 'store_type', 'type_y': 'holiday_type'}, inplace=True)

# Imputación de Petróleo: Continuidad de la señal económica
# Usamos ffill() para arrastrar el precio del viernes al fin de semana (lógica de mercado)
df['dcoilwtico'] = df['dcoilwtico'].ffill().bfill()

# Imputación de Feriados: Identificar días de operación normal (Workdays)
holiday_cols = ['holiday_type', 'locale', 'locale_name', 'description']
df[holiday_cols] = df[holiday_cols].fillna('Workday')
df['transferred'] = df['transferred'].fillna(False)

/tmp/ipykernel_33268/2716741715.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['transferred'] = df['transferred'].fillna(False)


In [13]:
# ==========================================
# 4. FEATURE ENGINEERING A: TIEMPO
# ==========================================
# Extraemos señales cíclicas de la fecha. Esto permite al modelo entender la estacionalidad.
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['is_quincena'] = df['date'].dt.day.isin([15, 30]) # Factor clave en el consumo de Ecuador

In [14]:
# ==========================================
# 5. FEATURE ENGINEERING B: LAGS Y ROLLING (Memoria)
# ==========================================
# ¡IMPORTANTE!: Los lags deben calcularse POR GRUPO (Tienda + Familia)
# Si no agrupamos, estarías asignando ventas de 'Leche' a 'Herramientas'.

#ESTO ES UNA SUERTE DE SUBGRUPOS

print("Generando Lags y Promedios Móviles... esto puede tardar un poco.")

# Lag de 1 día (Venta de ayer) y 7 días (Venta de la semana pasada)
#Los Lags permiten que el sistema "mire" su estado anterior para predecir el siguiente
#Un Lag (retraso) es simplemente desplazar los datos hacia adelante en el tiempo.
#Si hoy es martes, un Lag 1 es el valor del lunes. Un Lag 7 es el valor del martes de la semana pasada.
# Usamos transform para mantener el mismo número de filas
df['lag_1'] = df.groupby(['store_nbr', 'family'])['sales'].shift(1)
df['lag_7'] = df.groupby(['store_nbr', 'family'])['sales'].shift(7)
#Usa groupby, si solo haces .shift(1), la primera fila de la Tienda 2 (Leche)
#recibiría como "venta de ayer" la última fila de la Tienda 1 (Herramientas).

# Promedio Móvil (Rolling Mean): Captura la tendencia de la última semana
# min_periods=1 evita que perdamos demasiadas filas al inicio
df['rolling_sales_7'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean())
#x.rolling promedia las ventas de los últimos 7 días, suaviza singularidades y muestra la tendencia real

# Nota: Al crear lags de 7 días, las primeras 7 filas de cada grupo serán NaN.
# Las eliminamos para que el modelo no entrene con datos incompletos.
df.dropna(subset=['lag_1', 'lag_7'], inplace=True)

Generando Lags y Promedios Móviles... esto puede tardar un poco.


In [15]:
# ==========================================
# 6. SPLIT CRONOLÓGICO (Validation Strategy)
# ==========================================
# Respetamos la línea de tiempo para evitar Data Leakage.
df = df.sort_values('date')
fecha_corte = df['date'].max() - pd.Timedelta(days=15)

train_final = df[df['date'] < fecha_corte]
val_final = df[df['date'] >= fecha_corte]

In [16]:
# ==========================================
# 7. ENCODING Y SELECCIÓN FINAL
# ==========================================
# Definimos las variables que realmente aportan ROI al modelo
categorical_cols = ['family', 'city', 'state', 'store_type', 'holiday_type', 'locale']
numerical_cols = ['onpromotion', 'dcoilwtico', 'day_of_week', 'month', 'year',
                  'is_quincena', 'lag_1', 'lag_7', 'rolling_sales_7']

# One-Hot Encoding (OHE) aplicado solo al set de entrenamiento primero
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_final[categorical_cols].astype(str))

def prepare_X(df_subset, encoder, cat_cols, num_cols):
    """Función para estandarizar la transformación de datos"""
    encoded = encoder.transform(df_subset[cat_cols].astype(str))
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols), index=df_subset.index)
    return pd.concat([encoded_df, df_subset[num_cols]], axis=1)

X_train = prepare_X(train_final, encoder, categorical_cols, numerical_cols)
y_train = train_final['sales']

X_val = prepare_X(val_final, encoder, categorical_cols, numerical_cols)
y_val = val_final['sales']

print(f"Dataset listo. Dimensiones de X_train: {X_train.shape}")

Dataset listo. Dimensiones de X_train: (2959902, 96)


In [18]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error

# 1. Definimos los parámetros, MOVIENDO el early_stopping_rounds aquí
model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    tree_method='hist',
    random_state=42,
    objective='reg:squarederror',
    # --- NUEVA UBICACIÓN DE EARLY STOPPING ---
    early_stopping_rounds=50
)

# 2. El fit ahora es más limpio
# Solo pasamos los datos. El modelo ya sabe que debe vigilar el 'eval_set'
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], # Vigilamos solo el de validación para mayor velocidad
    verbose=100
)

# 3. Predicción y Evaluación
y_pred = model.predict(X_val)
y_pred = np.maximum(y_pred, 0) # Regla de negocio: ventas >= 0

rmsle = np.sqrt(mean_squared_log_error(y_val, y_pred))
print(f"\n--- Resultado de Validación ---")
print(f"RMSLE del Modelo: {rmsle:.4f}")

[0]	validation_0-rmse:1191.44001
[100]	validation_0-rmse:245.52702
[113]	validation_0-rmse:245.18487

--- Resultado de Validación ---
RMSLE del Modelo: 1.2878


In [20]:
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Cálculo de R2 (Coeficiente de Determinación)
# Nos indica qué porcentaje de la variación de las ventas es explicada por nuestro modelo.
r2 = r2_score(y_val, y_pred)

print(f"--- Métricas de Control ---")
print(f"R2 Score (Varianza explicada): {r2:.4f}")
print(f"RMSLE: {rmsle:.4f}")


--- Métricas de Control ---
R2 Score (Varianza explicada): 0.9620
RMSLE: 1.2878


In [22]:
# ==========================================
# 1. CARGA Y PREPARACIÓN DEL TEST (Data Integration)
# ==========================================
df_test_raw = pd.read_csv('/test.csv')
df_test_raw['date'] = pd.to_datetime(df_test_raw['date'])

# Guardamos los IDs para el archivo final (Metadata)
test_ids = df_test_raw['id']

# Unimos con información externa (Misma lógica que en Train)
df_test = df_test_raw.merge(stores, on='store_nbr', how='left')
df_test = df_test.merge(oil, on='date', how='left')
df_test = df_test.merge(holidays.drop_duplicates(subset=['date']), on='date', how='left')

# Limpieza y Renombramiento
df_test.rename(columns={'type_x': 'store_type', 'type_y': 'holiday_type'}, inplace=True)
df_test['dcoilwtico'] = df_test['dcoilwtico'].ffill().bfill()
df_test[holiday_cols] = df_test[holiday_cols].fillna('Workday')
df_test['transferred'] = df_test['transferred'].fillna(False)

# Feature Engineering: Tiempo
df_test['day_of_week'] = df_test['date'].dt.dayofweek
df_test['month'] = df_test['date'].dt.month
df_test['year'] = df_test['date'].dt.year
df_test['is_quincena'] = df_test['date'].dt.day.isin([15, 30])

# ==========================================
# 2. CÁLCULO DE LAGS (Continuidad de la Señal)
# ==========================================
# Para que el test tenga lags, concatenamos el final de train con el test
last_train_days = train_final.tail(30000) # Tomamos una muestra suficiente de los últimos días
combined = pd.concat([last_train_days, df_test], axis=0, sort=False)

# Recalculamos Lags y Rolling sobre el conjunto combinado
combined['lag_1'] = combined.groupby(['store_nbr', 'family'])['sales'].shift(1)
combined['lag_7'] = combined.groupby(['store_nbr', 'family'])['sales'].shift(7)
combined['rolling_sales_7'] = combined.groupby(['store_nbr', 'family'])['sales'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean())

# Extraemos solo las filas que pertenecen al TEST
df_test_final = combined[combined['sales'].isna()].copy()

# ==========================================
# 3. ENCODING Y PREDICCIÓN (Inference)
# ==========================================
# Usamos la función prepare_X que definimos anteriormente para mantener la consistencia
X_test = prepare_X(df_test_final, encoder, categorical_cols, numerical_cols)

# El modelo predice las ventas
y_test_pred = model.predict(X_test)

# Regla de Negocio: Las ventas no pueden ser negativas
y_test_pred = np.maximum(y_test_pred, 0)

# ==========================================
# 4. EXPORTACIÓN (Submission Format)
# ==========================================
submission = pd.DataFrame({
    'id': test_ids,
    'sales': y_test_pred
})

submission.to_csv('submission.csv', index=False)

print("¡Archivo 'submission.csv' creado con éxito!")
print(submission.head())

/tmp/ipykernel_33268/1544684654.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test['transferred'] = df_test['transferred'].fillna(False)


¡Archivo 'submission.csv' creado con éxito!
        id        sales
0  3000888    17.573927
1  3000889    14.207587
2  3000890    16.017790
3  3000891  1802.553589
4  3000892    14.207587
